# MATH 5010 Computer Lab — Section 3
## Joint and Conditional Probability

**Full-solution version**

This lab supports Section 3 of MATH 5010. The goal is to use Python to understand joint distributions, marginal distributions, conditional distributions, independence, covariance, total probability, Bayes theorem, and conditional independence.

By the end of this lab, students should be able to:

1. Construct joint PMFs from sample spaces.
2. Compute marginal and conditional distributions from a joint distribution.
3. Check independence numerically and symbolically.
4. Compute covariance and correlation from a joint distribution.
5. Simulate conditional distributions and verify formulas.
6. Use the law of total probability and Bayes theorem for random variables.
7. Understand conditional independence through a latent-variable example.

## 0. Python setup

We use `numpy` for simulation, `pandas` for tables, `scipy.stats` for probability distributions, and `matplotlib` for visualization.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import beta as beta_func

rng = np.random.default_rng(5010)

pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (6, 4)

## 1. Joint PMF from a sample space: two dice

Roll two fair dice. Let

$$
X = \text{first die}, \qquad Y = \text{sum of the two dice}.
$$

The sample space has 36 equally likely outcomes. We will build the joint PMF of $(X,Y)$ directly from the sample space.

In [ ]:
# Sample space for two fair dice
outcomes = [(i, j) for i in range(1, 7) for j in range(1, 7)]

records = []
for first, second in outcomes:
    x = first
    y = first + second
    records.append((x, y))

df = pd.DataFrame(records, columns=["X_first_die", "Y_sum"])

joint_counts = pd.crosstab(df["X_first_die"], df["Y_sum"])
joint_pmf = joint_counts / 36

joint_pmf

### Solution notes

The entry in row $x$ and column $y$ is

$$
P(X=x,Y=y).
$$

For example, $P(X=3,Y=8)=1/36$ because it corresponds to the outcome $(3,5)$.

In [ ]:
print("P(X=3, Y=8) =", joint_pmf.loc[3, 8])
print("This should equal 1/36 =", 1/36)

## 2. Marginal distributions

From a joint PMF, we recover marginal distributions by summing over the other variable:

$$
p_X(x)=\sum_y p_{X,Y}(x,y), \qquad p_Y(y)=\sum_x p_{X,Y}(x,y).
$$

In [ ]:
pmf_X = joint_pmf.sum(axis=1)
pmf_Y = joint_pmf.sum(axis=0)

print("Marginal PMF of X = first die")
display(pmf_X.to_frame("P(X=x)"))

print("Marginal PMF of Y = sum")
display(pmf_Y.to_frame("P(Y=y)"))

In [ ]:
fig, ax = plt.subplots()
pmf_Y.plot(kind="bar", ax=ax)
ax.set_xlabel("y")
ax.set_ylabel("P(Y=y)")
ax.set_title("Marginal PMF of Y = Sum of Two Dice")
plt.show()

### Exercise 2.1

Compute the following probabilities using the joint PMF:

$$
P(X=4), \qquad P(Y=7), \qquad P(X=4,Y=7), \qquad P(X=4 \mid Y=7).
$$

### Solution

Use marginalization and the definition of conditional probability:

$$
P(X=4\mid Y=7)=\frac{P(X=4,Y=7)}{P(Y=7)}.
$$

In [ ]:
p_X4 = pmf_X.loc[4]
p_Y7 = pmf_Y.loc[7]
p_X4_Y7 = joint_pmf.loc[4, 7]
p_X4_given_Y7 = p_X4_Y7 / p_Y7

print(f"P(X=4) = {p_X4:.4f}")
print(f"P(Y=7) = {p_Y7:.4f}")
print(f"P(X=4, Y=7) = {p_X4_Y7:.4f}")
print(f"P(X=4 | Y=7) = {p_X4_given_Y7:.4f}")

## 3. Conditional PMFs

The conditional PMF of $X$ given $Y=y$ is

$$
p_{X|Y}(x|y)=\frac{p_{X,Y}(x,y)}{p_Y(y)}, \qquad p_Y(y)>0.
$$

We compute the full conditional distribution $P(X=x\mid Y=7)$.

In [ ]:
y_value = 7
cond_X_given_Y7 = joint_pmf[y_value] / pmf_Y.loc[y_value]
cond_X_given_Y7 = cond_X_given_Y7[cond_X_given_Y7 > 0]
cond_X_given_Y7.to_frame("P(X=x | Y=7)")

In [ ]:
plt.figure()
plt.bar(cond_X_given_Y7.index, cond_X_given_Y7.values)
plt.xlabel("x")
plt.ylabel("P(X=x | Y=7)")
plt.title("Conditional PMF of X given Y=7")
plt.show()

### Exercise 3.1

Compute $P(Y=9\mid X=4)$ and interpret the answer.

### Solution

If the first die is 4, then $Y=9$ means the second die must be 5. Since the second die is fair, the answer should be $1/6$.

In [ ]:
p_Y9_given_X4 = joint_pmf.loc[4, 9] / pmf_X.loc[4]
print(f"P(Y=9 | X=4) = {p_Y9_given_X4:.4f}")
print("Expected answer: 1/6 =", 1/6)

## 4. Checking independence

Discrete random variables $X$ and $Y$ are independent if

$$
p_{X,Y}(x,y)=p_X(x)p_Y(y)
$$

for every pair $(x,y)$.

Here $X=$ first die and $Y=$ sum. These should **not** be independent because knowing the first die changes what sums are possible.

In [ ]:
# Create product of marginals table with the same shape as joint_pmf
prod_marginals = pd.DataFrame(
    np.outer(pmf_X.values, pmf_Y.values),
    index=pmf_X.index,
    columns=pmf_Y.index,
)

difference = joint_pmf - prod_marginals
max_abs_diff = np.abs(difference.values).max()

print("Maximum absolute difference between joint PMF and product of marginals:")
print(max_abs_diff)
print("If this were 0, X and Y would be independent. Here it is not 0.")

### Exercise 4.1

Let $A=$ first die and $B=$ second die. Are $A$ and $B$ independent?

### Solution

Yes. Since the dice are rolled independently,

$$
P(A=a,B=b)=\frac{1}{36}=\frac{1}{6}\frac{1}{6}=P(A=a)P(B=b).
$$

We verify this computationally.

In [ ]:
df_dice = pd.DataFrame(outcomes, columns=["A_first", "B_second"])
joint_AB = pd.crosstab(df_dice["A_first"], df_dice["B_second"]) / 36
pmf_A = joint_AB.sum(axis=1)
pmf_B = joint_AB.sum(axis=0)
prod_AB = pd.DataFrame(np.outer(pmf_A, pmf_B), index=pmf_A.index, columns=pmf_B.index)

print("Max absolute difference:", np.abs((joint_AB - prod_AB).values).max())
print("A and B are independent.")

## 5. Covariance and correlation

Covariance is

$$
\operatorname{Cov}(X,Y)=E[XY]-E[X]E[Y].
$$

Correlation is the standardized covariance:

$$
\rho_{X,Y}=\frac{\operatorname{Cov}(X,Y)}{\sqrt{\operatorname{Var}(X)}\sqrt{\operatorname{Var}(Y)}}.
$$

We compute these quantities for $X=$ first die and $Y=$ sum.

In [ ]:
def expectation_from_joint(joint, func):
    total = 0.0
    for x in joint.index:
        for y in joint.columns:
            total += func(x, y) * joint.loc[x, y]
    return total

EX = expectation_from_joint(joint_pmf, lambda x, y: x)
EY = expectation_from_joint(joint_pmf, lambda x, y: y)
EXY = expectation_from_joint(joint_pmf, lambda x, y: x*y)
EX2 = expectation_from_joint(joint_pmf, lambda x, y: x**2)
EY2 = expectation_from_joint(joint_pmf, lambda x, y: y**2)

varX = EX2 - EX**2
varY = EY2 - EY**2
covXY = EXY - EX*EY
corrXY = covXY / math.sqrt(varX * varY)

print(f"E[X] = {EX:.4f}")
print(f"E[Y] = {EY:.4f}")
print(f"Var(X) = {varX:.4f}")
print(f"Var(Y) = {varY:.4f}")
print(f"Cov(X,Y) = {covXY:.4f}")
print(f"Corr(X,Y) = {corrXY:.4f}")

### Solution interpretation

$Y=X+$ second die. Since $Y$ contains $X$, the covariance is positive. In fact,

$$
\operatorname{Cov}(X,Y)=\operatorname{Cov}(X,X+B)=\operatorname{Var}(X)+\operatorname{Cov}(X,B)=\operatorname{Var}(X),
$$

because the two dice are independent.

## 6. Uncorrelated does not imply independent

A common mistake is to think that covariance zero means independence. The implication

$$
X \perp Y \quad \Longrightarrow \quad \operatorname{Cov}(X,Y)=0
$$

is true when the relevant moments exist. The converse is false.

Example:

$$
X\sim \operatorname{Uniform}(-1,1), \qquad Y=X^2.
$$

Then $Y$ is completely determined by $X$, so $X$ and $Y$ are not independent. However, by symmetry,

$$
E[XY]=E[X^3]=0, \qquad E[X]=0,
$$

so $\operatorname{Cov}(X,Y)=0$.

In [ ]:
N = 200_000
X = rng.uniform(-1, 1, size=N)
Y = X**2

print("Sample covariance:", np.cov(X, Y, ddof=0)[0, 1])
print("Sample correlation:", np.corrcoef(X, Y)[0, 1])

plt.figure()
plt.scatter(X[:3000], Y[:3000], s=5, alpha=0.25)
plt.xlabel("X")
plt.ylabel("Y = X^2")
plt.title("Uncorrelated but not independent")
plt.show()

## 7. Continuous joint distribution: uniform on a triangle

Consider the joint density

$$
f_{X,Y}(x,y)=
\begin{cases}
2, & x\ge 0,\; y\ge 0,\; x+y\le 1,\\
0, & \text{otherwise.}
\end{cases}
$$

This is a valid PDF because the triangle has area $1/2$, so $2\cdot 1/2=1$.

The marginal density of $X$ is

$$
f_X(x)=\int_0^{1-x} 2\,dy=2(1-x), \qquad 0\le x\le 1.
$$

The conditional density of $Y$ given $X=x$ is

$$
f_{Y|X}(y|x)=\frac{2}{2(1-x)}=\frac{1}{1-x}, \qquad 0\le y\le 1-x.
$$

Therefore

$$
Y|X=x \sim \operatorname{Uniform}(0,1-x).
$$

In [ ]:
# Simulate from the triangle x>=0, y>=0, x+y<=1.
# Method: sample U,V uniform. If U+V>1, reflect to (1-U, 1-V).
N = 200_000
U = rng.uniform(size=N)
V = rng.uniform(size=N)
mask = U + V > 1
U[mask] = 1 - U[mask]
V[mask] = 1 - V[mask]
X_tri, Y_tri = U, V

plt.figure()
plt.scatter(X_tri[:5000], Y_tri[:5000], s=3, alpha=0.25)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Samples from uniform distribution on triangle")
plt.axis("equal")
plt.show()

In [ ]:
# Compare simulated marginal density of X with f_X(x)=2(1-x)
x_grid = np.linspace(0, 1, 200)
fx = 2 * (1 - x_grid)

plt.figure()
plt.hist(X_tri, bins=40, density=True, alpha=0.5, label="simulation")
plt.plot(x_grid, fx, label=r"$f_X(x)=2(1-x)$")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Marginal density of X")
plt.legend()
plt.show()

### Exercise 7.1

For the triangle distribution, compute

$$
P(X\le 0.4).
$$

### Solution

Using the marginal density:

$$
P(X\le 0.4)=\int_0^{0.4}2(1-x)\,dx
=2x-x^2\bigg|_0^{0.4}=0.8-0.16=0.64.
$$

In [ ]:
theory = 2*0.4 - 0.4**2
simulation = np.mean(X_tri <= 0.4)
print(f"Theory P(X <= 0.4) = {theory:.4f}")
print(f"Simulation estimate    = {simulation:.4f}")

### Exercise 7.2

For the triangle distribution, compute

$$
P(Y\le 0.2\mid X=0.5).
$$

### Solution

Given $X=x$, we have $Y|X=x\sim \operatorname{Uniform}(0,1-x)$. For $x=0.5$,

$$
Y|X=0.5\sim \operatorname{Uniform}(0,0.5).
$$

Thus

$$
P(Y\le 0.2\mid X=0.5)=\frac{0.2}{0.5}=0.4.
$$

In [ ]:
answer = 0.2 / (1 - 0.5)
print("P(Y <= 0.2 | X=0.5) =", answer)

## 8. Conditional expectation from a conditional distribution

For the triangle distribution,

$$
Y|X=x\sim \operatorname{Uniform}(0,1-x).
$$

Therefore,

$$
E[Y|X=x]=\frac{1-x}{2}.
$$

As a random variable,

$$
E[Y|X]=\frac{1-X}{2}.
$$

We verify this with simulation by binning values of $X$.

In [ ]:
bins = np.linspace(0, 1, 11)
bin_centers = (bins[:-1] + bins[1:]) / 2
conditional_means = []

for left, right in zip(bins[:-1], bins[1:]):
    idx = (X_tri >= left) & (X_tri < right)
    conditional_means.append(Y_tri[idx].mean())

conditional_means = np.array(conditional_means)
theory_means = (1 - bin_centers) / 2

comparison = pd.DataFrame({
    "x_bin_center": bin_centers,
    "simulated E[Y|X in bin]": conditional_means,
    "theory (1-x)/2": theory_means,
})
comparison

In [ ]:
plt.figure()
plt.plot(bin_centers, conditional_means, marker="o", label="simulation")
plt.plot(bin_centers, theory_means, marker="s", label=r"theory $(1-x)/2$")
plt.xlabel("x")
plt.ylabel("conditional mean")
plt.title(r"Conditional expectation $E[Y|X=x]$")
plt.legend()
plt.show()

## 9. Law of total probability for random variables

The law of total probability for random variables says:

Discrete conditioning variable:

$$
p_Y(y)=\sum_x p_{Y|X}(y|x)p_X(x).
$$

Continuous conditioning variable:

$$
f_Y(y)=\int f_{Y|X}(y|x) f_X(x)\,dx.
$$

We will study a classic example: **Poisson thinning**.

## 10. Poisson-Binomial example: thinning a Poisson count

Suppose

$$
X\sim \operatorname{Poisson}(\lambda),
\qquad
Y|X=x\sim \operatorname{Binomial}(x,p).
$$

Interpretation: $X$ is a total number of events, and each event is kept independently with probability $p$. Then $Y$ is the number of kept events.

By the law of total probability,

$$
P(Y=y)=\sum_{x=y}^{\infty}P(Y=y|X=x)P(X=x).
$$

The result is

$$
Y\sim \operatorname{Poisson}(\lambda p).
$$

In [ ]:
lam = 8
p = 0.35
N = 200_000

X_pois = rng.poisson(lam=lam, size=N)
Y_thinned = rng.binomial(X_pois, p)

# Empirical PMF
max_y = 12
values = np.arange(0, max_y + 1)
emp_pmf = np.array([np.mean(Y_thinned == y) for y in values])
theory_pmf = stats.poisson.pmf(values, mu=lam*p)

pmf_comparison = pd.DataFrame({
    "y": values,
    "simulation": emp_pmf,
    "Poisson(lambda*p) theory": theory_pmf,
})
pmf_comparison

In [ ]:
plt.figure()
plt.bar(values - 0.2, emp_pmf, width=0.4, label="simulation")
plt.bar(values + 0.2, theory_pmf, width=0.4, label=r"Poisson($\lambda p$)")
plt.xlabel("y")
plt.ylabel("probability")
plt.title("Poisson thinning: marginal distribution of Y")
plt.legend()
plt.show()

### Exercise 10.1

Let $X\sim \operatorname{Poisson}(10)$ and $Y|X=x\sim \operatorname{Binomial}(x,0.2)$. Find $E[Y]$ and $\operatorname{Var}(Y)$.

### Solution

Since $Y\sim \operatorname{Poisson}(\lambda p)=\operatorname{Poisson}(2)$,

$$
E[Y]=2, \qquad \operatorname{Var}(Y)=2.
$$

We can also get the mean by conditional expectation:

$$
E[Y]=E[E(Y|X)]=E[pX]=pE[X]=0.2(10)=2.
$$

In [ ]:
lam_ex = 10
p_ex = 0.2
print("E[Y] = lambda*p =", lam_ex * p_ex)
print("Var(Y) = lambda*p =", lam_ex * p_ex)

## 11. Bayes theorem for random variables: Beta-Bernoulli model

Let

$$
Y\sim \operatorname{Beta}(\alpha,\beta),
\qquad
X|Y=y\sim \operatorname{Bernoulli}(y).
$$

Then

$$
p(y|x) \propto p(x|y)p(y).
$$

If $x\in\{0,1\}$, then

$$
p(y|x) \propto y^x(1-y)^{1-x} y^{\alpha-1}(1-y)^{\beta-1}.
$$

Thus

$$
Y|X=x \sim \operatorname{Beta}(\alpha+x,\beta+1-x).
$$

In [ ]:
alpha, beta = 2, 3
x_observed = 1
posterior_alpha = alpha + x_observed
posterior_beta = beta + 1 - x_observed

print(f"Prior:     Beta({alpha}, {beta})")
print(f"Observed:  X = {x_observed}")
print(f"Posterior: Beta({posterior_alpha}, {posterior_beta})")

ys = np.linspace(0.001, 0.999, 400)
prior_pdf = stats.beta.pdf(ys, alpha, beta)
post_pdf = stats.beta.pdf(ys, posterior_alpha, posterior_beta)

plt.figure()
plt.plot(ys, prior_pdf, label=f"prior Beta({alpha},{beta})")
plt.plot(ys, post_pdf, label=f"posterior Beta({posterior_alpha},{posterior_beta})")
plt.xlabel("y")
plt.ylabel("density")
plt.title("Bayes theorem: Beta-Bernoulli update")
plt.legend()
plt.show()

### Exercise 11.1

Suppose $Y\sim \operatorname{Beta}(2,2)$ and conditional on $Y=y$, we observe three Bernoulli trials:

$$
x=(1,0,1).
$$

Find the posterior distribution of $Y$.

### Solution

There are $s=2$ successes and $m-s=1$ failures. With a Beta prior,

$$
Y|x \sim \operatorname{Beta}(\alpha+s,\beta+m-s)=\operatorname{Beta}(2+2,2+1)=\operatorname{Beta}(4,3).
$$

In [ ]:
alpha, beta = 2, 2
x_data = np.array([1, 0, 1])
s = x_data.sum()
m = len(x_data)
post_a = alpha + s
post_b = beta + m - s
print(f"Posterior distribution: Beta({post_a}, {post_b})")
print("Posterior mean:", post_a / (post_a + post_b))

## 12. Multivariate normal distribution

A bivariate normal random vector has mean vector

$$
\mu=\begin{pmatrix}\mu_X\\ \mu_Y\end{pmatrix}
$$

and covariance matrix

$$
\Sigma=\begin{pmatrix}
\sigma_X^2 & \rho\sigma_X\sigma_Y\\
\rho\sigma_X\sigma_Y & \sigma_Y^2
\end{pmatrix}.
$$

For a bivariate normal distribution, correlation controls the linear dependence. We simulate several correlations.

In [ ]:
mu = np.array([0, 0])
correlations = [-0.8, 0.0, 0.8]

for rho in correlations:
    Sigma = np.array([[1, rho], [rho, 1]])
    sample = rng.multivariate_normal(mu, Sigma, size=3000)
    plt.figure()
    plt.scatter(sample[:, 0], sample[:, 1], s=5, alpha=0.25)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(f"Bivariate normal with correlation rho={rho}")
    plt.axis("equal")
    plt.show()

### Exercise 12.1

Let

$$
\begin{pmatrix}X\\Y\end{pmatrix}
\sim N\left(
\begin{pmatrix}1\\2\end{pmatrix},
\begin{pmatrix}4&1.2\\1.2&9\end{pmatrix}
\right).
$$

Compute $\operatorname{Cov}(X,Y)$ and $\operatorname{Corr}(X,Y)$.

### Solution

The covariance is the off-diagonal entry:

$$
\operatorname{Cov}(X,Y)=1.2.
$$

The standard deviations are $\sigma_X=2$ and $\sigma_Y=3$. Therefore,

$$
\operatorname{Corr}(X,Y)=\frac{1.2}{2\cdot 3}=0.2.
$$

In [ ]:
Sigma = np.array([[4, 1.2], [1.2, 9]])
cov = Sigma[0, 1]
corr = cov / math.sqrt(Sigma[0, 0] * Sigma[1, 1])
print("Cov(X,Y) =", cov)
print("Corr(X,Y) =", corr)

## 13. Unit disk example

Let $(X,Y)$ be uniformly distributed over the unit disk

$$
D=\{(x,y):x^2+y^2\le 1\}.
$$

The joint density is

$$
f_{X,Y}(x,y)=\frac{1}{\pi}, \qquad (x,y)\in D.
$$

The marginal density of $X$ is

$$
f_X(x)=\int_{-\sqrt{1-x^2}}^{\sqrt{1-x^2}}\frac{1}{\pi}\,dy
=\frac{2}{\pi}\sqrt{1-x^2},\qquad -1\le x\le 1.
$$

Similarly,

$$
f_Y(y)=\frac{2}{\pi}\sqrt{1-y^2},\qquad -1\le y\le 1.
$$

The conditional density of $X$ given $Y=y$ is uniform on the horizontal slice:

$$
X|Y=y\sim \operatorname{Uniform}\left(-\sqrt{1-y^2},\sqrt{1-y^2}\right).
$$

In [ ]:
# Simulate uniformly from unit disk using polar coordinates
N = 200_000
R = np.sqrt(rng.uniform(size=N))
Theta = rng.uniform(0, 2*np.pi, size=N)
X_disk = R * np.cos(Theta)
Y_disk = R * np.sin(Theta)

plt.figure()
plt.scatter(X_disk[:5000], Y_disk[:5000], s=3, alpha=0.25)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Uniform samples from the unit disk")
plt.axis("equal")
plt.show()

In [ ]:
xgrid = np.linspace(-1, 1, 400)
fx_disk = 2/np.pi * np.sqrt(1 - xgrid**2)

plt.figure()
plt.hist(X_disk, bins=50, density=True, alpha=0.5, label="simulation")
plt.plot(xgrid, fx_disk, label=r"$f_X(x)=\frac{2}{\pi}\sqrt{1-x^2}$")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Marginal density of X for uniform disk")
plt.legend()
plt.show()

### Exercise 13.1

For the unit disk example, are $X$ and $Y$ independent?

### Solution

No. If $X$ and $Y$ were independent, the support would have to be a rectangle-type product support. But here knowing $Y=y$ changes the possible range of $X$:

$$
-\sqrt{1-y^2}\le X\le \sqrt{1-y^2}.
$$

For example, when $Y=0$, $X\in[-1,1]$; when $Y=0.9$, $X\in[-\sqrt{0.19},\sqrt{0.19}]$.

Also,

$$
f_{X,Y}(x,y)\ne f_X(x)f_Y(y)
$$

in general.

In [ ]:
# Check covariance: it is approximately zero by symmetry, even though X and Y are not independent.
cov_disk = np.cov(X_disk, Y_disk, ddof=0)[0, 1]
corr_disk = np.corrcoef(X_disk, Y_disk)[0, 1]
print("Sample Cov(X,Y):", cov_disk)
print("Sample Corr(X,Y):", corr_disk)
print("Covariance is approximately 0 by symmetry, but X and Y are not independent.")

## 14. Conditional independence

Two random variables $X$ and $Y$ are conditionally independent given $Z$ if

$$
p_{X,Y|Z}(x,y|z)=p_{X|Z}(x|z)p_{Y|Z}(y|z).
$$

Conditional independence does **not** mean ordinary independence.

We create a simple latent disease model:

- $Z=1$ means a person has a disease.
- $X=1$ means symptom 1 is present.
- $Y=1$ means symptom 2 is present.
- Conditional on disease status $Z$, the two symptoms are independent.

But marginally, $X$ and $Y$ are usually dependent because they share the common cause $Z$.

In [ ]:
# Latent disease model
p_Z1 = 0.10
p_X1_given_Z = {0: 0.05, 1: 0.80}
p_Y1_given_Z = {0: 0.10, 1: 0.70}

# Compute joint P(X,Y) by summing over Z
joint_XY = np.zeros((2, 2))
for z in [0, 1]:
    pz = p_Z1 if z == 1 else 1 - p_Z1
    for x in [0, 1]:
        px = p_X1_given_Z[z] if x == 1 else 1 - p_X1_given_Z[z]
        for y in [0, 1]:
            py = p_Y1_given_Z[z] if y == 1 else 1 - p_Y1_given_Z[z]
            joint_XY[x, y] += pz * px * py

joint_XY_df = pd.DataFrame(joint_XY, index=["X=0", "X=1"], columns=["Y=0", "Y=1"])
joint_XY_df

In [ ]:
pmf_X_latent = joint_XY.sum(axis=1)
pmf_Y_latent = joint_XY.sum(axis=0)
prod_latent = np.outer(pmf_X_latent, pmf_Y_latent)

print("P(X=1,Y=1) =", joint_XY[1,1])
print("P(X=1)P(Y=1) =", pmf_X_latent[1] * pmf_Y_latent[1])
print("Difference =", joint_XY[1,1] - pmf_X_latent[1] * pmf_Y_latent[1])
print("So X and Y are not marginally independent.")

### Exercise 14.1

Verify conditional independence for $Z=1$ by checking

$$
P(X=1,Y=1|Z=1)=P(X=1|Z=1)P(Y=1|Z=1).
$$

### Solution

Given $Z=1$, the model says the symptoms are conditionally independent. Therefore,

$$
P(X=1,Y=1|Z=1)=0.80\cdot 0.70=0.56.
$$

In [ ]:
lhs = p_X1_given_Z[1] * p_Y1_given_Z[1]
rhs = p_X1_given_Z[1] * p_Y1_given_Z[1]
print("P(X=1,Y=1 | Z=1) =", lhs)
print("P(X=1 | Z=1) P(Y=1 | Z=1) =", rhs)

## 15. Practice problems with complete solutions

The following problems summarize the main computational skills of Section 3.

### Practice Problem 1: Joint PMF table

Suppose $(X,Y)$ has joint PMF:

|       | $Y=0$ | $Y=1$ | $Y=2$ |
|------:|------:|------:|------:|
| $X=0$ | 0.10  | 0.20  | 0.10  |
| $X=1$ | 0.05  | 0.25  | 0.30  |

1. Find the marginal PMFs of $X$ and $Y$.
2. Find $P(Y=2|X=1)$.
3. Are $X$ and $Y$ independent?

### Solution

Use row sums and column sums. Then check whether $p(x,y)=p_X(x)p_Y(y)$ for all pairs.

In [ ]:
joint = pd.DataFrame(
    [[0.10, 0.20, 0.10],
     [0.05, 0.25, 0.30]],
    index=[0, 1],
    columns=[0, 1, 2],
)

pmf_X_pr = joint.sum(axis=1)
pmf_Y_pr = joint.sum(axis=0)
cond_Y2_given_X1 = joint.loc[1, 2] / pmf_X_pr.loc[1]
prod = pd.DataFrame(np.outer(pmf_X_pr, pmf_Y_pr), index=joint.index, columns=joint.columns)
independent = np.allclose(joint.values, prod.values)

print("Joint PMF:")
display(joint)
print("Marginal PMF of X:")
display(pmf_X_pr.to_frame("P(X=x)"))
print("Marginal PMF of Y:")
display(pmf_Y_pr.to_frame("P(Y=y)"))
print("P(Y=2 | X=1) =", cond_Y2_given_X1)
print("Are X and Y independent?", independent)
print("Max difference from product of marginals:", np.abs((joint - prod).values).max())

### Practice Problem 2: Covariance from a joint PMF

Using the joint PMF in Practice Problem 1, compute

$$
E[X],\quad E[Y],\quad E[XY],\quad \operatorname{Cov}(X,Y),\quad \operatorname{Corr}(X,Y).
$$

### Solution

Use

$$
E[g(X,Y)]=\sum_x\sum_y g(x,y)p(x,y).
$$

In [ ]:
def E_joint_table(joint_table, func):
    total = 0.0
    for x in joint_table.index:
        for y in joint_table.columns:
            total += func(x, y) * joint_table.loc[x, y]
    return total

EX = E_joint_table(joint, lambda x, y: x)
EY = E_joint_table(joint, lambda x, y: y)
EXY = E_joint_table(joint, lambda x, y: x*y)
EX2 = E_joint_table(joint, lambda x, y: x**2)
EY2 = E_joint_table(joint, lambda x, y: y**2)

varX = EX2 - EX**2
varY = EY2 - EY**2
covXY = EXY - EX*EY
corrXY = covXY / math.sqrt(varX * varY)

print(f"E[X] = {EX:.4f}")
print(f"E[Y] = {EY:.4f}")
print(f"E[XY] = {EXY:.4f}")
print(f"Var(X) = {varX:.4f}")
print(f"Var(Y) = {varY:.4f}")
print(f"Cov(X,Y) = {covXY:.4f}")
print(f"Corr(X,Y) = {corrXY:.4f}")

### Practice Problem 3: Continuous conditional distribution

Let

$$
f_{X,Y}(x,y)=2,\qquad 0<x<1,\;0<y<1-x.
$$

Find $f_X(x)$ and $f_{Y|X}(y|x)$.

### Solution

The support implies $0<x<1$ and $0<y<1-x$. Thus

$$
f_X(x)=\int_0^{1-x}2\,dy=2(1-x),\qquad 0<x<1.
$$

Then

$$
f_{Y|X}(y|x)=\frac{f_{X,Y}(x,y)}{f_X(x)}
=\frac{2}{2(1-x)}=\frac{1}{1-x},
$$

for $0<y<1-x$.

In [ ]:
# Numerical check of f_X normalization
xgrid = np.linspace(0, 1, 10001)
fx = 2*(1-xgrid)
area = np.trapz(fx, xgrid)
print("Integral of f_X over [0,1] ≈", area)

### Practice Problem 4: Law of total probability

Let $X\sim \operatorname{Poisson}(6)$ and $Y|X=x\sim \operatorname{Binomial}(x,0.4)$.

Find the marginal distribution of $Y$, then compute $P(Y=3)$.

### Solution

By Poisson thinning,

$$
Y\sim \operatorname{Poisson}(6\cdot 0.4)=\operatorname{Poisson}(2.4).
$$

Therefore,

$$
P(Y=3)=e^{-2.4}\frac{2.4^3}{3!}.
$$

In [ ]:
mu_y = 6 * 0.4
p_y3 = stats.poisson.pmf(3, mu_y)
print("Y ~ Poisson(2.4)")
print("P(Y=3) =", p_y3)

### Practice Problem 5: Bayes theorem for Beta-Bernoulli

Let $P\sim\operatorname{Beta}(3,5)$. Conditional on $P=p$, we observe $n=10$ independent Bernoulli trials with $s=7$ successes.

Find the posterior distribution of $P$ and the posterior mean.

### Solution

The Beta prior is conjugate to the Bernoulli/binomial likelihood:

$$
P|\text{data}\sim \operatorname{Beta}(\alpha+s,\beta+n-s)
=\operatorname{Beta}(3+7,5+3)=\operatorname{Beta}(10,8).
$$

The posterior mean is

$$
E[P|\text{data}]=\frac{10}{10+8}=\frac{5}{9}\approx 0.5556.
$$

In [ ]:
alpha, beta = 3, 5
n, s = 10, 7
post_alpha = alpha + s
post_beta = beta + n - s
post_mean = post_alpha / (post_alpha + post_beta)
print(f"Posterior: Beta({post_alpha}, {post_beta})")
print("Posterior mean:", post_mean)

## 16. Lab checklist

After completing this lab, you should be able to do the following in Python:

- Build a joint PMF from data or a sample space.
- Compute marginal and conditional PMFs.
- Check independence by comparing the joint PMF to the product of marginals.
- Compute covariance and correlation.
- Simulate from a continuous joint distribution.
- Compute conditional distributions from joint densities.
- Use the law of total probability for a hierarchical model.
- Use Bayes theorem to update a prior distribution.
- Explain the difference between independence and conditional independence.